In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

In [3]:
!pip install -q datasets huggingface_hub hf_transfer

In [4]:
import polars as pl
from huggingface_hub import login, hf_hub_download

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Fixed Re-Merge Pipeline

## What the old merge did wrong

The original merge notebook:
1. Stripped `t1_` from ALL `parent_id` values (including `t3_` ones)
2. Left-joined against comments to get `parent_body`
3. **Filtered out rows where `parent_body` was null** — this killed ALL top-level comments (whose parent is a post, not a comment) and all comments whose parent was deleted

This destroyed thread completeness for AITA, CMV, and UNPOPULAR.

## What this notebook does instead
1. For **top-level comments** (`parent_id` starts with `t3_`): uses the post's `selftext` as `parent_body`
2. For **reply comments** (`parent_id` starts with `t1_`): looks up the parent comment's body as before
3. **Samples whole threads** (all comments for selected posts) to preserve tree integrity


In [5]:
# ══════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════
import os
REPO_ID = "KS-AO-HUB5/dissent-data"

SUBREDDITS_TO_FIX = [
    "amitheasshole",
    "changemyview",
    "unpopularopinion",
    "the10thdentist",
    "politicalopinions",
]

# Target ~600K comments per subreddit to stay near 3M total
TARGET_COMMENTS_PER_SUB = 600_000
MIN_THREAD_COMMENTS = 5

OUTPUT_DIR = "/content/drive/MyDrive/My_Dissent_project/merged_subreddits"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Repo: {REPO_ID}")
print(f"Subreddits to fix: {SUBREDDITS_TO_FIX}")
print(f"Target per sub: {TARGET_COMMENTS_PER_SUB:,}")
print(f"Output: {OUTPUT_DIR}")

Repo: KS-AO-HUB5/dissent-data
Subreddits to fix: ['amitheasshole', 'changemyview', 'unpopularopinion', 'the10thdentist', 'politicalopinions']
Target per sub: 600,000
Output: /content/drive/MyDrive/My_Dissent_project/merged_subreddits


In [ ]:
from huggingface_hub import login
login(token="") # REPLACE

In [7]:
# ══════════════════════════════════════════════════════════════════════
# Download all needed files
# ══════════════════════════════════════════════════════════════════════

downloaded = {}

for sub in SUBREDDITS_TO_FIX:
    print(f"\n📥 Downloading r/{sub}...")

    if sub == "amitheasshole":
        posts_file = "amitheasshole_posts_v2.csv"
        comments_file = "amitheasshole_comments_v2.csv"

    else:
        posts_file = f"{sub}_posts.csv"
        comments_file = f"{sub}_comments.csv"

    posts_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=posts_file,
    )

    comments_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=comments_file,
    )

    downloaded[sub] = {
        "posts": posts_path,
        "comments": comments_path,
    }

    # Quick row counts
    n_posts = pl.scan_csv(posts_path).select(pl.len()).collect().item()
    n_comments = pl.scan_csv(comments_path).select(pl.len()).collect().item()
    print(f"   Posts: {n_posts:,}")
    print(f"   Comments: {n_comments:,}")

print("\n✅ All downloads complete.")


📥 Downloading r/amitheasshole...


amitheasshole_posts_v2.csv:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

amitheasshole_comments_v2.csv:   0%|          | 0.00/25.4G [00:00<?, ?B/s]

   Posts: 683,844
   Comments: 64,998,000

📥 Downloading r/changemyview...


changemyview_posts.csv:   0%|          | 0.00/253M [00:00<?, ?B/s]

...

the10thdentist_comments.csv:   0%|          | 0.00/518M [00:00<?, ?B/s]

   Posts: 22,879
   Comments: 1,672,414

📥 Downloading r/politicalopinions...


politicalopinions_posts.csv:   0%|          | 0.00/11.1M [00:00<?, ?B/s]

politicalopinions_comments.csv:   0%|          | 0.00/59.1M [00:00<?, ?B/s]

   Posts: 5,008
   Comments: 90,172

✅ All downloads complete.


In [8]:
# ══════════════════════════════════════════════════════════════════════
# RAM-safe merge: Sample threads FIRST, then join only what we need
# ══════════════════════════════════════════════════════════════════════

import gc

summary = []

for sub in SUBREDDITS_TO_FIX:
    print(f"\n{'=' * 70}")
    print(f"  Processing r/{sub}")
    print(f"{'=' * 70}")

    paths = downloaded[sub]

    # ────────────────────────────────────────────────────────
    # Step 1: Read ONLY the columns we need (lazy scan)
    # ────────────────────────────────────────────────────────
    print("  Step 1: Scanning files...")

    comments_lf = pl.scan_csv(paths["comments"]).select([
        "id", "body", "author", "score", "created_utc",
        "link_id", "parent_id", "author_flair_text", "edited", "controversiality"
    ])

    posts_lf = pl.scan_csv(paths["posts"]).select([
        "id", "author", "created_utc", "num_comments",
        "edited", "score", "title", "selftext"
    ])

    # ────────────────────────────────────────────────────────
    # Step 2: CHEAPLY count comments per thread (no join yet)
    #         Then pick threads BEFORE the expensive join
    # ────────────────────────────────────────────────────────
    print("  Step 2: Counting comments per thread (cheap aggregation)...")

    thread_sizes = (
        comments_lf
        .with_columns(
            pl.col("link_id").str.replace(r"^t3_", "").alias("post_id")
        )
        .filter(
            (pl.col("body").is_not_null()) &
            (~pl.col("body").is_in(["[deleted]", "[removed]", ""]))
        )
        .group_by("post_id")
        .agg(pl.len().alias("n_comments"))
        .filter(pl.col("n_comments") >= MIN_THREAD_COMMENTS)
        .collect()
    )

    print(f"    Eligible threads: {thread_sizes.height:,}")
    print(f"    Total eligible comments: {thread_sizes['n_comments'].sum():,}")

    # ── Sample threads ──
    if thread_sizes["n_comments"].sum() <= TARGET_COMMENTS_PER_SUB:
        selected_posts = thread_sizes.select("post_id")
        print(f"    Taking ALL eligible threads")
    else:
        shuffled = thread_sizes.sample(fraction=1.0, shuffle=True, seed=42)
        selected_posts = (
            shuffled
            .with_columns(pl.col("n_comments").cum_sum().alias("_running"))
            .filter(pl.col("_running") <= TARGET_COMMENTS_PER_SUB)
            .select("post_id")
        )
        print(f"    Selected {selected_posts.height:,} threads")

    selected_post_set = set(selected_posts["post_id"].to_list())
    print(f"    Target post_ids: {len(selected_post_set):,}")

    del thread_sizes
    if 'shuffled' in locals():
        del shuffled
    gc.collect()

    # ────────────────────────────────────────────────────────
    # Step 3: Collect ONLY comments belonging to selected threads
    #         This is the RAM-saving step!
    # ────────────────────────────────────────────────────────
    print("  Step 3: Collecting only selected-thread comments into RAM...")

    comments_subset = (
        comments_lf
        .with_columns(
            pl.col("link_id").str.replace(r"^t3_", "").alias("post_id")
        )
        .filter(pl.col("post_id").is_in(selected_post_set))
        .collect()
    )
    print(f"    Comments loaded: {comments_subset.height:,}")

    # ────────────────────────────────────────────────────────
    # Step 4: Collect ONLY the posts we need
    # ────────────────────────────────────────────────────────
    print("  Step 4: Collecting selected posts...")

    posts_subset = (
        posts_lf
        .filter(pl.col("id").is_in(selected_post_set))
        .collect()
    )
    print(f"    Posts loaded: {posts_subset.height:,}")

    # ────────────────────────────────────────────────────────
    # Step 5: NOW do the joins — on a small subset only
    # ────────────────────────────────────────────────────────
    print("  Step 5: Joining (on subset only — RAM safe)...")

    # Rename columns for clarity
    comments_df = comments_subset.rename({
        "id": "comment_id",
        "body": "comment_body",
        "author": "comment_author",
        "score": "comment_score",
        "created_utc": "comment_created_utc",
        "author_flair_text": "comment_author_flair_text",
        "edited": "comment_edited",
        "controversiality": "comment_controversiality",
    })

    posts_df = posts_subset.rename({
        "author": "post_author",
        "created_utc": "post_created_utc",
        "edited": "post_edited",
        "score": "post_score",
        "title": "post_title",
        "selftext": "post_selftext",
    }).rename({"id": "post_id"})

    # Add helper columns
    comments_df = comments_df.with_columns([
        pl.col("parent_id").str.starts_with("t3_").alias("_is_top_level"),
        pl.when(pl.col("parent_id").str.starts_with("t1_"))
          .then(pl.col("parent_id").str.replace(r"^t1_", ""))
          .otherwise(pl.lit(None))
          .alias("_parent_cid"),
    ])

    # Parent comment body lookup (from the SAME subset — cheap!)
    parent_lookup = comments_df.select([
        pl.col("comment_id").alias("_parent_cid"),
        pl.col("comment_body").alias("_parent_comment_body"),
    ])

    # Join posts
    result = comments_df.join(
        posts_df,
        on="post_id",
        how="left",
    )

    del comments_df, posts_df, comments_subset, posts_subset
    gc.collect()

    # Join parent comments
    result = result.join(
        parent_lookup,
        on="_parent_cid",
        how="left",
    )

    del parent_lookup
    gc.collect()

    # Build parent_body
    result = result.with_columns([
        pl.when(pl.col("_is_top_level"))
          .then(pl.col("post_selftext"))
          .otherwise(pl.col("_parent_comment_body"))
          .alias("parent_body")
    ]).drop(["_parent_cid", "_parent_comment_body", "post_selftext", "_is_top_level"])

    # ────────────────────────────────────────────────────────
    # Step 6: Final filter (comment_body & parent_body valid)
    # ────────────────────────────────────────────────────────
    print("  Step 6: Final filter...")

    result = result.filter(
        (pl.col("comment_body").is_not_null()) &
        (~pl.col("comment_body").is_in(["[deleted]", "[removed]", ""])) &
        (pl.col("parent_body").is_not_null()) &
        (~pl.col("parent_body").is_in(["[deleted]", "[removed]", ""]))
    )

    n_threads = result["post_id"].n_unique()
    n_comments = result.height
    n_top_level = result.filter(pl.col("parent_id").str.starts_with("t3_")).height
    n_replies = result.filter(pl.col("parent_id").str.starts_with("t1_")).height

    print(f"\n  ✅ r/{sub} result:")
    print(f"    Threads:    {n_threads:,}")
    print(f"    Comments:   {n_comments:,}")
    print(f"    Top-level:  {n_top_level:,} ({n_top_level/max(n_comments,1)*100:.1f}%)")
    print(f"    Replies:    {n_replies:,} ({n_replies/max(n_comments,1)*100:.1f}%)")

    # ── Tree completeness check ──
    all_ids = set(result["comment_id"].to_list())
    reply_parents = (
        result
        .filter(pl.col("parent_id").str.starts_with("t1_"))
        .select(pl.col("parent_id").str.replace(r"^t1_", "").alias("pid"))
    )
    n_reply_total = reply_parents.height
    n_found = sum(1 for pid in reply_parents["pid"].to_list() if pid in all_ids)
    n_orphaned = n_reply_total - n_found
    orphan_pct = n_orphaned / max(n_reply_total, 1) * 100

    print(f"\n    Tree completeness:")
    print(f"      Parent found: {n_found:,} / {n_reply_total:,}")
    print(f"      Orphaned: {n_orphaned:,} ({orphan_pct:.1f}%)")

    # ── Save ──
    output_path = os.path.join(OUTPUT_DIR, f"{sub}.csv")
    result = result.sort("link_id", "comment_created_utc")
    result.write_csv(output_path)
    file_size_mb = os.path.getsize(output_path) / 1e6
    print(f"\n    💾 Saved: {output_path} ({file_size_mb:.0f} MB)")

    summary.append({
        "subreddit": sub,
        "threads": n_threads,
        "comments": n_comments,
        "top_level": n_top_level,
        "replies": n_replies,
        "orphan_pct": round(orphan_pct, 1),
    })

    del result
    gc.collect()

# ── Final summary ──
print(f"\n\n{'=' * 70}")
print(f"  SUMMARY")
print(f"{'=' * 70}")

total_comments = 0
for s in summary:
    print(f"\n  r/{s['subreddit']:25s} {s['threads']:>8,} threads  {s['comments']:>10,} comments  "
          f"top-level: {s['top_level']:>8,}  orphans: {s['orphan_pct']:.1f}%")
    total_comments += s['comments']

print(f"\n  TOTAL new comments: {total_comments:,}")
print(f"  + T10D (1.66M) + POL-OP (88K) = ~{total_comments + 1_656_000 + 88_000:,}")


  Processing r/amitheasshole
  Step 1: Scanning files...
  Step 2: Counting comments per thread (cheap aggregation)...
    Eligible threads: 658,264
    Total eligible comments: 64,902,510
    Selected 5,893 threads
    Target post_ids: 5,893
  Step 3: Collecting only selected-thread comments into RAM...
    Comments loaded: 599,836
  Step 4: Collecting selected posts...
    Posts loaded: 5,893
  Step 5: Joining (on subset only — RAM safe)...
  Step 6: Final filter...

  ✅ r/amitheasshole result:
    Threads:    5,893
    Comments:   586,657
    Top-level:  326,105 (55.6%)
    Replies:    260,552 (44.4%)

    Tree completeness:
      Parent found: 256,330 / 260,552
      Orphaned: 4,222 (1.6%)

    💾 Saved: /content/drive/MyDrive/My_Dissent_project/merged_subreddits/amitheasshole.csv (1052 MB)

  Processing r/changemyview
  Step 1: Scanning files...
  Step 2: Counting comments per thread (cheap aggregation)...
    Eligible threads: 112,798
    Total eligible comments: 12,640,295
    S

In [9]:
# ══════════════════════════════════════════════════════════════════════
# Verify: spot-check one subreddit
# ══════════════════════════════════════════════════════════════════════

check_sub = "unpopularopinion"
check_path = os.path.join(OUTPUT_DIR, f"{check_sub}.csv")

if not os.path.exists(check_path):
    print(f"⚠️  {check_path} not found — did the merge complete?")
else:
    df = pl.read_csv(check_path, n_rows=20, infer_schema_length=10000)
    print(f"Columns: {df.columns}")
    print(f"\nFirst 5 rows:")
    print(df.head(5))

    # Check parent_id distribution
    df_full = pl.scan_csv(check_path, infer_schema_length=10000)
    parent_type_counts = (
        df_full
        .with_columns([
            pl.when(pl.col("parent_id").str.starts_with("t3_"))
              .then(pl.lit("top_level"))
            .when(pl.col("parent_id").str.starts_with("t1_"))
              .then(pl.lit("reply"))
            .otherwise(pl.lit("other"))
              .alias("parent_type")
        ])
        .group_by("parent_type")
        .agg(pl.len().alias("count"))
        .collect()
    )
    print(f"\nParent type distribution:")
    print(parent_type_counts)

    # Check that parent_body is non-null for top-level comments
    top_level_null_check = (
        df_full
        .filter(pl.col("parent_id").str.starts_with("t3_"))
        .select([
            pl.len().alias("total_top_level"),
            pl.col("parent_body").is_null().sum().alias("null_parent_body"),
        ])
        .collect()
    )
    print(f"\nTop-level comments parent_body null check:")
    print(top_level_null_check)

Columns: ['comment_id', 'comment_body', 'comment_author', 'comment_score', 'comment_created_utc', 'link_id', 'parent_id', 'comment_author_flair_text', 'comment_edited', 'comment_controversiality', 'post_id', 'post_author', 'post_created_utc', 'num_comments', 'post_edited', 'post_score', 'post_title', 'parent_body']

First 5 rows:
shape: (5, 18)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ comment_i ┆ comment_b ┆ comment_a ┆ comment_s ┆ … ┆ post_edit ┆ post_scor ┆ post_titl ┆ parent_b │
│ d         ┆ ody       ┆ uthor     ┆ core      ┆   ┆ ed        ┆ e         ┆ e         ┆ ody      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ i64       ┆   ┆ str       ┆ i64       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ j2erpjh   ┆ The       ┆ Desperate ┆ 5        

In [10]:
# ══════════════════════════════════════════════════════════════════════
# Cost estimate for Gemini batch labeling
# ══════════════════════════════════════════════════════════════════════
import os
import polars as pl

OUTPUT_DIR = "/content/drive/MyDrive/My_Dissent_project/merged_subreddits"
total_requests = 0
total_input_chars = 0

for f in sorted(os.listdir(OUTPUT_DIR)):
    if not f.endswith(".csv"):
        continue
    path = os.path.join(OUTPUT_DIR, f)
    df = pl.scan_csv(path, infer_schema_length=10000)

    stats = df.select([
        pl.len().alias("n_rows"),
        pl.col("comment_body").cast(pl.Utf8).str.len_chars().clip(0, 4000).mean().alias("avg_comment_chars"),
        pl.col("parent_body").cast(pl.Utf8).str.len_chars().clip(0, 4000).mean().alias("avg_parent_chars"),
    ]).collect()

    n = stats["n_rows"].item()
    avg_comment = stats["avg_comment_chars"].item() or 0
    avg_parent = stats["avg_parent_chars"].item() or 0
    avg_input_tokens = (avg_comment + avg_parent) / 4 + 180  # +180 for system prompt + formatting

    total_requests += n
    total_input_chars += n * (avg_comment + avg_parent)

    print(f"  {f:30s}  {n:>10,} rows  avg_comment={avg_comment:.0f} chars  avg_parent={avg_parent:.0f} chars  "
          f"avg_input={avg_input_tokens:.0f} tokens/req")

avg_tokens_overall = total_input_chars / total_requests / 4 + 180
input_cost = total_requests * avg_tokens_overall * 0.05 / 1_000_000
output_cost = total_requests * 20 * 0.20 / 1_000_000
total_cost = input_cost + output_cost

print(f"\n{'═' * 60}")
print(f"  Total requests:  {total_requests:,}")
print(f"  Avg input tokens: {avg_tokens_overall:.0f}")
print(f"  Input cost:  ${input_cost:.2f}")
print(f"  Output cost: ${output_cost:.2f}")
print(f"  ─────────────────")
print(f"  TOTAL ESTIMATED: ${total_cost:.2f}")
print(f"{'═' * 60}")

if total_cost > 180:
    print(f"\n  ⚠️  This is close to your $200 budget. Consider reducing data.")
else:
    print(f"\n  ✅ Within $200 budget (${200 - total_cost:.2f} headroom)")

  amitheasshole.csv                  586,657 rows  avg_comment=272 chars  avg_parent=1283 chars  avg_input=569 tokens/req
  changemyview.csv                   577,825 rows  avg_comment=470 chars  avg_parent=743 chars  avg_input=483 tokens/req
  politicalopinions.csv               86,493 rows  avg_comment=533 chars  avg_parent=849 chars  avg_input=526 tokens/req
  the10thdentist.csv                 594,278 rows  avg_comment=201 chars  avg_parent=532 chars  avg_input=363 tokens/req
  unpopularopinion.csv               580,579 rows  avg_comment=196 chars  avg_parent=466 chars  avg_input=345 tokens/req

════════════════════════════════════════════════════════════
  Total requests:  2,425,832
  Avg input tokens: 443
  Input cost:  $53.74
  Output cost: $9.70
  ─────────────────
  TOTAL ESTIMATED: $63.44
════════════════════════════════════════════════════════════

  ✅ Within $200 budget ($136.56 headroom)
